# Can we predict `passed_quiz`?
Follows `protocol.yaml`: group holdout on `student_key` (`holdout.csv`), 5-fold GroupKFold CV on the training rows, selection by mean CV ROC AUC, holdout scored once.

**Feature timing.** The quiz closes the session, so `session_minutes` (length of the whole session) is only final once the quiz is over; it is excluded. `hints_used` accrues during the session before the quiz and is kept. `student_key` is excluded per the protocol.

In [ ]:
import json, hashlib, numpy as np, pandas as pd
from sklearn.model_selection import GroupKFold
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.dummy import DummyClassifier
from sklearn.metrics import roc_auc_score
from sklearn.base import clone

assert hashlib.sha256(open('data.csv','rb').read()).hexdigest()=='5a0b0221ed641db40322e1a95cae32cc67d3525ecb500813f95d6c109b2a7409'
data=pd.read_csv('data.csv')
hold_idx=pd.read_csv('holdout.csv')['row'].to_numpy()   # 0-based positions in data.csv
is_hold=np.zeros(len(data),bool); is_hold[hold_idx]=True
train=data[~is_hold]; hold=data[is_hold]
assert set(train.student_key).isdisjoint(hold.student_key)
print(len(train),len(hold),train.passed_quiz.mean())

In [ ]:
TARGET='passed_quiz'
FEATURES=['enrol_year','school_code','prior_score','hints_used']   # session_minutes excluded: not known before the quiz ends
Xtr,ytr,gtr=train[FEATURES],train[TARGET],train.student_key
Xho,yho=hold[FEATURES],hold[TARGET]

def prep_lr():
    return ColumnTransformer([('num',StandardScaler(),['enrol_year','prior_score','hints_used']),
                              ('sch',OneHotEncoder(handle_unknown='ignore',min_frequency=20),['school_code'])])
# all candidates named before any is scored
candidates={
 'baseline_prior':Pipeline([('m',DummyClassifier(strategy='prior'))]),
 'logreg':Pipeline([('p',prep_lr()),('m',LogisticRegression(C=1.0,max_iter=1000))]),
 'logreg_C0.1':Pipeline([('p',prep_lr()),('m',LogisticRegression(C=0.1,max_iter=1000))]),
 'hgb':Pipeline([('m',HistGradientBoostingClassifier(max_depth=3,learning_rate=0.05,max_iter=200,random_state=0))]),
}
print(list(candidates))

In [ ]:
gkf=GroupKFold(n_splits=5)
folds=list(gkf.split(Xtr,ytr,gtr))
cv_scores={}
for name,est in candidates.items():
    s=[]
    for tr,va in folds:
        m=clone(est).fit(Xtr.iloc[tr],ytr.iloc[tr])
        s.append(roc_auc_score(ytr.iloc[va],m.predict_proba(Xtr.iloc[va])[:,1]))
    cv_scores[name]=s
cv=pd.DataFrame({k:{'mean':np.mean(v),'std':np.std(v)} for k,v in cv_scores.items()}).T
cv

In [ ]:
chosen=max(cv_scores,key=lambda k:np.mean(cv_scores[k]))
final=clone(candidates[chosen]).fit(Xtr,ytr)
p=final.predict_proba(Xho)[:,1]
holdout_auc=roc_auc_score(yho,p)
print('chosen:',chosen,'CV AUC %.3f'%np.mean(cv_scores[chosen]),'| holdout AUC %.3f'%holdout_auc)

In [ ]:
tr_pos=np.flatnonzero(~is_hold)
evidence={'features':FEATURES,'candidates':list(candidates),
 'cv_folds':[[tr_pos[a].tolist(),tr_pos[b].tolist()] for a,b in folds],
 'cv_scores':cv_scores,'chosen':chosen,
 'holdout_rows':hold_idx.tolist(),'holdout_scores':{chosen:holdout_auc}}
json.dump(evidence,open('evidence.json','w'))

## Conclusion
The chosen candidate's holdout ROC AUC (printed above) is the headline. Signal is modest: `hints_used` is the main predictor available before the quiz. `session_minutes` correlates with passing but is not known at prediction time, so it is not used.